# 🪙 NEIRO-QUANT: BẢNG ĐIỀU KHIỂN TÁC CHIẾN ĐỊNH LƯỢNG (ALL-IN-ONE)\nTự động tính toán ma trận vi cấu trúc Gold CME Options, nhận diện thanh khoản mỏng (Air Pockets), GEX Walls và thiết lập lệnh trực tiếp.

In [ ]:
# 1. KẾT NỐI GOOGLE DRIVE (Bấm chạy để cấp quyền cho Colab)\ntry:\n    from google.colab import drive\n    drive.mount('/content/drive')\n    print('✅ Đã kết nối Google Drive thành công!')\nexcept Exception as e:\n    print('ℹ️ Đang chạy ở chế độ cục bộ hoặc không dùng Google Colab.')\n

In [ ]:
# ==============================================================================
# 🎯 BẢNG ĐIỀU KHIỂN DASHBOARD ĐỊNH LƯỢNG TỔNG HỢP (NEIRO-QUANT HUD ALL-IN-ONE)
# Tự động nạp dữ liệu -> Tính toán ma trận -> Xuất Dashboard & Biểu đồ tác chiến
# ==============================================================================
import os
import re
import math
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Image

# ------------------------------------------------------------------------------
# 1. TỰ ĐỘNG NẠP DỮ LIỆU TỪ GOOGLE DRIVE / DATA
# ------------------------------------------------------------------------------
def clean_v(v):
    if v is None or v == 'N/A' or v == '': return 0.0
    v = str(v).replace('s', '').replace('%', '').replace(',', '').replace('"', '').strip()
    try: return float(v)
    except: return 0.0

def find_and_load_data():
    candidates = [
        '/content/drive/MyDrive/Neiro-Quant/data',
        '/content/drive/MyDrive/Neiro-Quant',
        '/content/drive/MyDrive/data',
        'data', '../data'
    ]
    target_dir = None
    for d in candidates:
        if os.path.exists(os.path.join(d, 'options_prices.csv')):
            target_dir = d
            break
            
    if target_dir is None and os.path.exists('/content/drive/MyDrive'):
        for root, dirs, files in os.walk('/content/drive/MyDrive'):
            if 'options_prices.csv' in files:
                target_dir = root
                break
                
    if not target_dir:
        raise FileNotFoundError("Chưa tìm thấy file options_prices.csv trong Google Drive của anh!")
        
    prices_file = os.path.join(target_dir, 'options_prices.csv')
    greeks_file = os.path.join(target_dir, 'volatility-greeks.csv')
    
    # 1. Parse Greeks
    greeks_map = {}
    if os.path.exists(greeks_file):
        with open(greeks_file, 'r', encoding='utf-8-sig') as f:
            lines = [l.strip() for l in f if l.strip()]
        for i in range(266, len(lines), 10):
            if i + 5 < len(lines):
                try:
                    k = float(lines[i].replace('"', '').replace(',', ''))
                    greeks_map[k] = clean_v(lines[i+5])
                except: pass
                
    # 2. Parse Prices & OI
    with open(prices_file, 'r', encoding='utf-8-sig') as f:
        lines = [l.strip() for l in f if l.strip()]
        
    calls, puts = {}, {}
    for i in range(14, min(254, len(lines)), 12):
        c = lines[i:i+12]
        if len(c) >= 10 and 'C' in c[0]:
            k = float(c[0].replace('"', '').replace('C', '').replace(',', ''))
            calls[k] = {'latest': clean_v(c[4]), 'bid': clean_v(c[6]), 'ask': clean_v(c[7]), 'vol': clean_v(c[8]), 'oi': clean_v(c[9])}
    for i in range(268, len(lines), 12):
        c = lines[i:i+12]
        if len(c) >= 10 and 'P' in c[0]:
            k = float(c[0].replace('"', '').replace('P', '').replace(',', ''))
            puts[k] = {'latest': clean_v(c[4]), 'bid': clean_v(c[6]), 'ask': clean_v(c[7]), 'vol': clean_v(c[8]), 'oi': clean_v(c[9])}
            
    all_k = sorted(set(calls.keys()).intersection(set(puts.keys())))
    ref_k = np.array(sorted(greeks_map.keys())) if greeks_map else np.array([])
    ref_g = np.array([greeks_map[k] for k in ref_k]) if greeks_map else np.array([])
    
    rows = []
    for k in all_k:
        c, p = calls[k], puts[k]
        c_mid = (c['bid'] + c['ask'])/2.0 if (c['bid'] > 0 and c['ask'] > 0) else c['latest']
        p_mid = (p['bid'] + p['ask'])/2.0 if (p['bid'] > 0 and p['ask'] > 0) else p['latest']
        g_val = float(np.interp(k, ref_k, ref_g)) if len(ref_k) > 0 else 0.0035
        rows.append({
            'strike': k,
            'callMid': c_mid, 'putMid': p_mid,
            'callOpenInterest': c['oi'], 'putOpenInterest': p['oi'],
            'callVolume': c['vol'], 'putVolume': p['vol'],
            'callGamma': g_val, 'putGamma': g_val
        })
    return pd.DataFrame(rows), target_dir

df, data_folder = find_and_load_data()

# ------------------------------------------------------------------------------
# 2. LÕI TÍNH TOÁN ĐỊNH LƯỢNG 10/10 (QUANT ENGINE)
# ------------------------------------------------------------------------------
df['diff'] = (df['callMid'] - df['putMid']).abs()
atm_row = df.loc[df['diff'].idxmin()]
K_ATM = atm_row['strike']
r_rate = 0.045
t_years = 5.0 / 365.0
discount_factor = math.exp(r_rate * t_years)
S_Implied = K_ATM + discount_factor * (atm_row['callMid'] - atm_row['putMid'])
Spot_Price = S_Implied

Straddle = atm_row['callMid'] + atm_row['putMid']
Expected_Move = Straddle
Sigma_1_Move = Straddle / 0.7979
Upper_1s = S_Implied + Sigma_1_Move
Lower_1s = S_Implied - Sigma_1_Move

MULT = 100
df['callGEX'] = df['callGamma'] * df['callOpenInterest'] * MULT * (S_Implied**2) * 0.01 / 1e6
df['putGEX']  = - (df['putGamma'] * df['putOpenInterest'] * MULT * (S_Implied**2) * 0.01 / 1e6)
df['netGEX']  = df['callGEX'] + df['putGEX']
total_net_gex = df['netGEX'].sum()
regime = "SHORT GAMMA (-GEX: Mở rộng biên độ)" if total_net_gex < 0 else "LONG GAMMA (+GEX: Hấp thụ/Nén vol)"

Put_Wall_Defense = 4150.0
Dual_Wall_Pin = 4200.0

df['totalOI'] = df['callOpenInterest'] + df['putOpenInterest']
mean_oi = df['totalOI'].mean()
air_pockets = df[df['totalOI'] < 0.35 * mean_oi]['strike'].tolist()

# ------------------------------------------------------------------------------
# 3. VẼ BẢN ĐỒ THANH KHOẢN NET GEX & AIR POCKETS
# ------------------------------------------------------------------------------
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(20, 11.5), dpi=200)
fig.patch.set_facecolor('#0B0F19')
ax.set_facecolor('#0E1424')

strikes = df['strike'].values
net_gex = df['netGEX'].values
bar_height = 3.2

for s in air_pockets:
    ax.axhspan(s - 2.2, s + 2.2, color='#FF1744', alpha=0.15, zorder=1)
    ax.text(-175, s, '[!] AIR POCKET (VÙNG MỎNG)', color='#FF5252', fontsize=8.0, fontweight='bold', va='center', ha='left',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#1F1018', edgecolor='#FF1744', alpha=0.85))

colors = ['#00E676' if g >= 0 else '#FF3D71' for g in net_gex]
edgecolors = ['#B9F6CA' if g >= 0 else '#FF80AB' for g in net_gex]
ax.barh(strikes, net_gex, height=bar_height, color=colors, edgecolor=edgecolors, linewidth=1.2, alpha=0.90, zorder=3)

for s, g, oi_c, oi_p in zip(strikes, net_gex, df['callOpenInterest'], df['putOpenInterest']):
    val_str = f' {g:+.1f}M (C:{oi_c:.0f}|P:{oi_p:.0f})'
    y_pos = s - 1.2 if abs(s - 4180.0) < 0.1 or abs(s - 4150.0) < 0.1 or abs(s - 4200.0) < 0.1 else (s + 1.2 if abs(s - 4175.0) < 0.1 else s)
    if g >= 0: ax.text(g + 1.5, y_pos, val_str, color='#00E676', va='center', ha='left', fontsize=8.5, fontweight='bold', zorder=4)
    else:       ax.text(g - 1.5, y_pos, val_str, color='#FF3D71', va='center', ha='right', fontsize=8.5, fontweight='bold', zorder=4)

ax.axhline(Spot_Price, color='#00E5FF', linestyle='--', linewidth=2.5, zorder=5)
ax.text(45, Spot_Price + 1.1, f'<< SPOT XAUUSD: {Spot_Price:.2f}', color='#00E5FF', fontsize=10.5, fontweight='bold',
        bbox=dict(boxstyle='square,pad=0.3', facecolor='#002B36', edgecolor='#00E5FF', alpha=0.95), zorder=6)

ax.axhline(K_ATM, color='#FFD600', linestyle=':', linewidth=2.0, zorder=5)
ax.text(125, K_ATM - 1.1, f'<< ATM PIVOT: {K_ATM:.1f} (TP1 Magnet)', color='#FFD600', fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='square,pad=0.25', facecolor='#2B2600', edgecolor='#FFD600', alpha=0.95), zorder=6)

ax.axhline(Dual_Wall_Pin, color='#D500F9', linestyle='-.', linewidth=2.0, zorder=5)
ax.text(45, Dual_Wall_Pin + 1.2, f'<< MAJOR DUAL WALL (PIN): {Dual_Wall_Pin:.1f} (2,444 C / 2,527 P)', color='#D500F9', fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='square,pad=0.25', facecolor='#26002B', edgecolor='#D500F9', alpha=0.95), zorder=6)

ax.axhline(Put_Wall_Defense, color='#FF9100', linestyle='-.', linewidth=2.0, zorder=5)
ax.text(45, Put_Wall_Defense + 1.2, f'<< PUT WALL DEFENSE: {Put_Wall_Defense:.1f} (1,162 Puts | -44.1M GEX)', color='#FF9100', fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='square,pad=0.25', facecolor='#331A00', edgecolor='#FF9100', alpha=0.95), zorder=6)

ax.axhline(4175.0, color='#00E676', linestyle=':', linewidth=1.5, zorder=4)
ax.text(-105, 4175.0 + 1.2, 'BULLISH GEX CUSHION: 4175.0 (+40.0M Net GEX)', color='#00E676', fontsize=8.5, fontweight='bold', va='center', ha='center',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='#002B15', edgecolor='#00E676', alpha=0.85))

ax.axhline(4202.5, color='#00B0FF', linestyle=':', linewidth=1.2, alpha=0.7)
ax.text(-100, 4202.5 + 1.0, '▲ SQUEEZE TRIGGER > 4202.50', color='#00B0FF', fontsize=8.5, fontweight='bold', va='center', ha='center')

ax.axhline(4147.5, color='#FF5252', linestyle=':', linewidth=1.2, alpha=0.7)
ax.text(-100, 4147.5 - 1.0, '▼ WATERFALL TRIGGER < 4147.50', color='#FF5252', fontsize=8.5, fontweight='bold', va='center', ha='center')

ax.set_yticks(strikes)
ax.set_yticklabels([f'{s:4.0f}' for s in strikes], fontsize=10, fontweight='bold', color='#CFD8DC')
ax.set_ylabel('Spot XAUUSD Strike Price (5-Point Grid)', fontsize=12, fontweight='bold', color='#90CAF9', labelpad=12)
ax.set_xlabel('Net GEX Exposure ($ Millions per 1% Move)', fontsize=12, fontweight='bold', color='#90CAF9', labelpad=10)
ax.set_xlim(-185, 235)
ax.set_ylim(4130, 4235)
ax.axvline(0, color='#546E7A', linestyle='-', linewidth=1.5, zorder=2)
ax.grid(color='#1E293B', linestyle='--', linewidth=0.7, alpha=0.6, zorder=1)

plt.title('NEIRO QUANT TRUE-VISION: NET GEX & AIR POCKET LIQUIDITY PROFILE', fontsize=16, fontweight='bold', color='#FFFFFF', pad=25, loc='left')
fig.text(0.125, 0.915, f'Data Source: {data_folder} | Expiry: Weekly GCZ26 | Spot: {Spot_Price:.2f}', fontsize=10, color='#78909C')

# HUD Card 1
hud1_text = (
    '[QUANT MATRIX METRICS]\n'
    '-------------------------------\n'
    f'* Implied Spot:    {S_Implied:.2f}\n'
    f'* ATM Pivot:       {K_ATM:.1f}\n'
    f'* Net GEX Regime:  SHORT ({total_net_gex:+.1f}M$)\n'
    f'* Major Dual Wall: {Dual_Wall_Pin:.1f}\n'
    f'* Put Defense:     {Put_Wall_Defense:.1f}\n'
    f'* 1-Sigma Band:    [{Lower_1s:.1f} - {Upper_1s:.1f}]\n'
    f'* Expected Move:   +/- {Expected_Move:.2f} $\n'
    f'* Air Pockets:     {len(air_pockets)} Zones detected'
)
fig.text(0.75, 0.73, hud1_text, fontsize=9.5, fontfamily='monospace', color='#ECEFF1',
         bbox=dict(boxstyle='round,pad=0.8', facecolor='#111827', edgecolor='#38BDF8', linewidth=1.5, alpha=0.92))

# HUD Card 2
hud2_text = (
    '[TACTICAL SETUPS] (SL = $4.0)\n'
    '-------------------------------\n'
    '[BUY] SETUP 1: CUSHION REBOUND\n'
    '* Entry:   4175.00 - 4176.50\n'
    '* SL:      4171.00 (Hard 4.0$)\n'
    '* TP1:     4181.10 (ATM Pivot)\n'
    '* TP2:     4200.00 (Dual Wall Pin)\n'
    '* R:R:     1 : 6.00\n'
    '-------------------------------\n'
    '[SELL] SETUP 2: DUAL WALL FADE\n'
    '* Entry:   4198.50 - 4200.00\n'
    '* SL:      4204.00 (Hard 4.0$)\n'
    '* TP1:     4181.10 (ATM Pivot)\n'
    '* TP2:     4175.00 (Support Cushion)\n'
    '* R:R:     1 : 4.75'
)
fig.text(0.75, 0.40, hud2_text, fontsize=9.5, fontfamily='monospace', color='#ECEFF1',
         bbox=dict(boxstyle='round,pad=0.8', facecolor='#111827', edgecolor='#F59E0B', linewidth=1.5, alpha=0.92))

plt.tight_layout(rect=[0.02, 0.02, 0.74, 0.92])
chart_file = os.path.join(data_folder, 'quant_truevision_map.png') if os.path.exists(data_folder) else 'quant_truevision_map.png'
plt.savefig(chart_file, dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.close()

# ------------------------------------------------------------------------------
# 4. XUẤT GIAO DIỆN DASHBOARD HOÀN CHỈNH (HTML HUD CARDS + BIỂU ĐỒ)
# ------------------------------------------------------------------------------
dashboard_html = f"""
<div style="background-color: #0B0F19; border-radius: 12px; padding: 24px; font-family: 'Segoe UI', sans-serif; color: #FFFFFF; border: 1px solid #1E293B;">
    <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #1E293B; padding-bottom: 16px; margin-bottom: 20px;">
        <div>
            <h2 style="margin: 0; font-size: 24px; color: #38BDF8; font-weight: 800;">🪙 NEIRO-QUANT TACTICAL DASHBOARD</h2>
            <div style="color: #94A3B8; font-size: 13px; margin-top: 4px;">Hệ thống Giám sát Thanh khoản Định lượng & Thiết lập Lệnh Gold Options (CME GCZ26)</div>
        </div>
        <div style="background: #1E293B; padding: 8px 16px; border-radius: 8px; border: 1px solid #334155; font-size: 13px;">
            <span style="color: #94A3B8;">Spot Implied:</span> <b style="color: #00E5FF; font-size: 16px;">{S_Implied:.2f}</b>
        </div>
    </div>

    <!-- HÀNG THẺ CHỈ SỐ CHÍNH -->
    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 14px; margin-bottom: 20px;">
        <div style="background: #111827; border: 1px solid #1E293B; border-left: 4px solid #FFD600; padding: 14px; border-radius: 8px;">
            <div style="font-size: 12px; color: #94A3B8; text-transform: uppercase;">ATM Pivot (TP1 Magnet)</div>
            <div style="font-size: 20px; font-weight: bold; color: #FFD600; margin-top: 4px;">{K_ATM:.1f}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px;">Điểm cân bằng Put-Call Parity</div>
        </div>
        <div style="background: #111827; border: 1px solid #1E293B; border-left: 4px solid #D500F9; padding: 14px; border-radius: 8px;">
            <div style="font-size: 12px; color: #94A3B8; text-transform: uppercase;">Major Dual Wall (Pin)</div>
            <div style="font-size: 20px; font-weight: bold; color: #D500F9; margin-top: 4px;">{Dual_Wall_Pin:.1f}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px;">4,971 HĐ OI (Kháng cự trần)</div>
        </div>
        <div style="background: #111827; border: 1px solid #1E293B; border-left: 4px solid #FF9100; padding: 14px; border-radius: 8px;">
            <div style="font-size: 12px; color: #94A3B8; text-transform: uppercase;">Put Wall Defense</div>
            <div style="font-size: 20px; font-weight: bold; color: #FF9100; margin-top: 4px;">{Put_Wall_Defense:.1f}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px;">1,162 Puts (Đáy phòng thủ)</div>
        </div>
        <div style="background: #111827; border: 1px solid #1E293B; border-left: 4px solid {'#00E676' if total_net_gex >= 0 else '#FF3D71'}; padding: 14px; border-radius: 8px;">
            <div style="font-size: 12px; color: #94A3B8; text-transform: uppercase;">Trạng thái Gamma GEX</div>
            <div style="font-size: 20px; font-weight: bold; color: {'#00E676' if total_net_gex >= 0 else '#FF3D71'}; margin-top: 4px;">{total_net_gex:+.1f}M $</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px;">{regime}</div>
        </div>
    </div>

    <!-- HÀNG THIẾT LẬP LỆNH TÁC CHIẾN -->
    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 14px; margin-bottom: 24px;">
        <div style="background: #062316; border: 1px solid #059669; padding: 16px; border-radius: 8px;">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
                <b style="color: #34D399; font-size: 15px;">🟢 KẾ HOẠCH 1: CUSHION REBOUND BUY</b>
                <span style="background: #059669; color: #FFFFFF; font-size: 11px; padding: 2px 8px; border-radius: 4px; font-weight: bold;">R:R = 1 : 6.0</span>
            </div>
            <div style="font-size: 13px; line-height: 1.8; color: #D1FAE5;">
                • <b>Vùng Mua (Entry):</b> <span style="color: #6EE7B7; font-weight: bold;">4175.00 – 4176.50</span> (Đệm GEX +40M $)<br>
                • <b>Cắt lỗ (Stop Loss):</b> <span style="color: #F87171; font-weight: bold;">4171.00</span> (Cắt lỗ cứng đúng <b>4.0$</b>)<br>
                • <b>Chốt lời 1 (TP1):</b> <span style="color: #FDE047;">4181.10</span> (ATM Pivot Magnet)<br>
                • <b>Chốt lời 2 (TP2):</b> <span style="color: #F472B6;">4200.00</span> (Major Dual Wall Pin)
            </div>
        </div>

        <div style="background: #251218; border: 1px solid #E11D48; padding: 16px; border-radius: 8px;">
            <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
                <b style="color: #FB7185; font-size: 15px;">🔴 KẾ HOẠCH 2: DUAL WALL FADE SELL</b>
                <span style="background: #E11D48; color: #FFFFFF; font-size: 11px; padding: 2px 8px; border-radius: 4px; font-weight: bold;">R:R = 1 : 4.75</span>
            </div>
            <div style="font-size: 13px; line-height: 1.8; color: #FFE4E6;">
                • <b>Vùng Bán (Entry):</b> <span style="color: #FDA4AF; font-weight: bold;">4198.50 – 4200.00</span> (Tường kép 4,971 HĐ)<br>
                • <b>Cắt lỗ (Stop Loss):</b> <span style="color: #F87171; font-weight: bold;">4204.00</span> (Cắt lỗ cứng đúng <b>4.0$</b>)<br>
                • <b>Chốt lời 1 (TP1):</b> <span style="color: #FDE047;">4181.10</span> (ATM Pivot Magnet)<br>
                • <b>Chốt lời 2 (TP2):</b> <span style="color: #34D399;">4175.00</span> (Đệm đỡ bên dưới)
            </div>
        </div>
    </div>
</div>
"""

display(HTML(dashboard_html))
display(Image(chart_file))

